In [ ]:
!pip install pandas plotly dash

In [ ]:
import pandas as pd
import plotly.express as px
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

In [ ]:
url = 'https://raw.githubusercontent.com/datasets/covid-19/main/data/time-series-19-covid-combined.csv'
df = pd.read_csv(url)

In [ ]:
print(df.head())

         Date Country/Region Province/State  Confirmed  Recovered  Deaths
0  2020-01-22    Afghanistan            NaN          0        0.0       0
1  2020-01-23    Afghanistan            NaN          0        0.0       0
2  2020-01-24    Afghanistan            NaN          0        0.0       0
3  2020-01-25    Afghanistan            NaN          0        0.0       0
4  2020-01-26    Afghanistan            NaN          0        0.0       0


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

df = df.drop_duplicates()

df = df.fillna(0)

In [ ]:
print("Total Confirmed:", df['Confirmed'].sum())
print("Total Deaths:", df['Deaths'].sum())
print("Total Recovered:", df['Recovered'].sum())

Total Confirmed: 118939403514
Total Deaths: 2261860890
Total Recovered: 23227207579.0


In [ ]:
trend = df.groupby('Date')[['Confirmed','Deaths','Recovered']].sum().reset_index()

fig = px.line(trend, x='Date', y=['Confirmed','Deaths','Recovered'],
              title='COVID Trend Over Time')

fig.show()

In [ ]:
country = df.groupby('Country/Region')[['Confirmed']].sum().reset_index()

fig = px.bar(country, x='Country/Region', y='Confirmed',
             title='Country Cases')

fig.show()

In [ ]:
latest = df.iloc[-1]

fig = px.pie(values=[latest['Confirmed'], latest['Deaths'], latest['Recovered']],
             names=['Confirmed','Deaths','Recovered'],
             title='Case Distribution')

fig.show()

In [ ]:
fig = px.scatter(df, x='Confirmed', y='Deaths',
                 size='Recovered', color='Country/Region',
                 title='Confirmed vs Deaths')

fig.show()

In [ ]:
app = dash.Dash(__name__)

countries = df['Country/Region'].unique()

app.layout = html.Div([
    html.H1("COVID-19 Dashboard"),

    dcc.Dropdown(
        id='country',
        options=[{'label': i, 'value': i} for i in countries],
        value=countries[0]
    ),

    dcc.Graph(id='graph')
])

In [ ]:
@app.callback(
    Output('graph', 'figure'),
    Input('country', 'value')
)
def update_graph(selected_country):
    filtered = df[df['Country/Region'] == selected_country]

    fig = px.line(filtered, x='Date', y='Confirmed',
                  title=f"{selected_country} Cases")

    return fig

In [ ]:
if __name__ == '__main__':
    app.run()

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
